In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
env_file = next((p / "env.env" for p in [ROOT, *ROOT.parents] if (p / "env.env").exists()), None)
if env_file:
    load_dotenv(env_file)

In [ ]:
import inspect

import torch
from torch.nn import functional as F

from adaloop import api

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

In [ ]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

### Step-1: - Setup Model 

In [ ]:
cfg = api.load_config(ROOT / "configs/mvp.yaml")
vlm = api.load_model(cfg)
model, processor = vlm.model, vlm.processor
language_model = model.model.language_model
print("Device map:", model.hf_device_map)

In [ ]:
MAX_NEW_TOKENS = 1024

#### Step-2: - Understanding Model Architecture

In [ ]:
state_hf = model.state_dict()
for k,v in state_hf.items(): 
    print(k,v.shape)

In [ ]:
vision_tower = model.model.vision_tower

In [ ]:
print(model.config.vision_config)

In [ ]:
for name, module in vision_tower.named_children():
    print(name, type(module).__name__)

In [ ]:
print(inspect.getsource(type(vision_tower).forward))

In [ ]:
print(inspect.getsource(type(model.model).forward))

#### Sample Input 

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
question = "What animal is on the candy?"

image = api.load_image(url, cfg["res_px"])
inputs = api.prepare_inputs(vlm, question, image)

for key, value in inputs.items():
    print(key, value.shape)

In [ ]:
print(inspect.getsource(type(model.model).get_image_features))

#### Extract projected visual tokens

In [ ]:
visual_tokens, vision_outputs = api.extract_visual_tokens(vlm, inputs)

print("Image resolution:", image.size)
print("Image grid:", inputs["image_grid_thw"])
print("Visual tokens:", visual_tokens.shape)
print("Vision hidden states:", vision_outputs.last_hidden_state.shape)

In [ ]:
image_mask = inputs["input_ids"] == model.config.image_token_id
num_placeholders = image_mask.sum().item()

print("Image placeholders:", num_placeholders)
print("Projected visual tokens:", visual_tokens.shape[0])

assert num_placeholders == visual_tokens.shape[0]
assert visual_tokens.shape[-1] == 2304

### Step-4: - Token pruning 

In [ ]:
for k in [1.0, 0.5, 0.25]:
    indices = api.random_selector(visual_tokens, k)
    pruned = api.prune_inputs(vlm, inputs, vision_outputs, indices)
    print(f"k={k} | Visual tokens={len(indices)} | Total tokens={pruned['input_ids'].shape[-1]}")

In [ ]:
api.set_loops(vlm, 3)

pruned = api.prune_inputs(vlm, inputs, vision_outputs, api.random_selector(visual_tokens, k=1.0))
original = api.last_logits(vlm, inputs)
reconstructed = api.last_logits(vlm, pruned)

print("Max difference:", (original - reconstructed).abs().max().item())
print("Same predicted token:", torch.equal(original.argmax(-1), reconstructed.argmax(-1)))

In [ ]:
api.set_loops(vlm, 3)

original_tokens = api.generate(vlm, inputs, MAX_NEW_TOKENS).token_ids
pruned = api.prune_inputs(vlm, inputs, vision_outputs, api.random_selector(visual_tokens, k=1.0))
reconstructed_tokens = api.generate(vlm, pruned, MAX_NEW_TOKENS).token_ids

print("Identical generation:", torch.equal(original_tokens, reconstructed_tokens))

In [ ]:
for k in [1.0, 0.5, 0.25]:
    indices = api.random_selector(visual_tokens, k=k)
    pruned = api.prune_inputs(vlm, inputs, vision_outputs, indices)

    print(f"\nRetention: {k * 100}%")
    print(f"Visual tokens: {len(indices)}")
    print(f"Decoder input tokens: {pruned['input_ids'].shape[-1]}")
    print(f"Output: {api.generate(vlm, pruned, MAX_NEW_TOKENS).text}")

In [ ]:
def decoder_hook(module, args, kwargs):
    print("Decoder input:", kwargs["inputs_embeds"].shape)

handle = language_model.register_forward_pre_hook(decoder_hook, with_kwargs=True)
try:
    indices = api.random_selector(visual_tokens, k=0.5)
    pruned = api.prune_inputs(vlm, inputs, vision_outputs, indices)
    api.generate(vlm, pruned, max_new_tokens=3)
finally:
    handle.remove()

#### Uniform Pruning

In [ ]:
api.set_loops(vlm, 3)

for k in [1.0, 0.5, 0.25]:
    indices = api.uniform_selector(inputs["image_grid_thw"], k)
    pruned = api.prune_inputs(vlm, inputs, vision_outputs, indices)
    print(f"\nRetention: {k * 100}%")
    print(f"Visual tokens: {len(indices)}")
    print(api.generate(vlm, pruned, MAX_NEW_TOKENS).text)

#### Training-free text-guided pruning

In [ ]:
text_embeddings = api.question_embeddings(vlm, question)

print("Text:", text_embeddings.shape)
print("Visual:", visual_tokens.shape)

In [ ]:
api.set_loops(vlm, 3)

for k in [1.0, 0.5, 0.25]:
    indices, scores = api.text_mean_selector(visual_tokens, text_embeddings, k)
    pruned = api.prune_inputs(vlm, inputs, vision_outputs, indices)
    print(f"\nRetention: {k * 100}%")
    print("Visual tokens:", len(indices))
    print(api.generate(vlm, pruned, MAX_NEW_TOKENS).text)

In [ ]:
import matplotlib.pyplot as plt
_, h, w = inputs["image_grid_thw"][0].tolist()

plt.imshow(scores.reshape(h, w).detach().cpu(),cmap="viridis")
plt.colorbar(label="Cosine similarity")
plt.title(question)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_relevance(image, question, scores, image_grid_thw):
    _, h, w = image_grid_thw[0].tolist()

    heatmap = scores.detach().float().cpu().reshape(h, w).numpy()

    # Match the processed image grid
    img = image.resize((w * 40, h * 40))

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(img)
    axes[0].set_title("Original image")

    axes[1].imshow(heatmap, cmap="jet", interpolation="nearest")
    axes[1].set_title(f"Cosine similarity ({h} × {w})")

    axes[2].imshow(img)
    overlay = axes[2].imshow(
        heatmap,
        cmap="jet",
        alpha=0.45,
        interpolation="nearest",
        extent=(0, img.width, img.height, 0)
    )
    axes[2].set_title("Question relevance overlay")

    for ax in axes:
        ax.axis("off")

    fig.colorbar(overlay, ax=axes, shrink=0.7,
                 label="Cosine similarity")

    fig.suptitle(question, fontsize=14)
    plt.show()

In [ ]:
questions = [
    "What animal is on the candy?",
    "animal",
    "orange candy",
    "green candy",
    "hand",
    "background",
]

for q in questions:
    _, scores = api.text_mean_selector(visual_tokens, api.question_embeddings(vlm, q), k=0.5)
    visualize_relevance(image, q, scores, inputs["image_grid_thw"])

In [ ]:
attention = language_model.layers[0].self_attn

print(inspect.getsource(type(attention).forward))
print("Attention backend:",
      language_model.config._attn_implementation)

In [ ]:
api.set_loops(vlm, 1)

with torch.inference_mode():
    outputs = model(**inputs, output_attentions=True, use_cache=False, logits_to_keep=1)

attentions = outputs.attentions

if attentions is None:
    print("Attention weights unavailable")
else:
    for i, attn in enumerate(attentions):
        print(i, attn.shape if attn is not None else None)

#### Decoder based pruning

In [ ]:
image_positions = (
    inputs["input_ids"][0] == model.config.image_token_id
).nonzero(as_tuple=True)[0]

ids = inputs["input_ids"][0].tolist()

q_ids = processor.tokenizer(
    question, add_special_tokens=False
)["input_ids"]

matches = [
    i for i in range(len(ids) - len(q_ids) + 1)
    if ids[i:i + len(q_ids)] == q_ids
    and i > image_positions[-1].item()
]

assert matches, "Question token sequence not found"

start = matches[-1]
question_positions = torch.arange(start, start + len(q_ids))

print("Image tokens:", len(image_positions))
print("Question tokens:", len(question_positions))
print("Question:", processor.decode(ids[start:start + len(q_ids)]))

In [ ]:
def attention_relevance(attention, question_positions, image_positions):
    A = attention[0].float()

    A = A.index_select(1, question_positions.to(A.device))
    A = A.index_select(2, image_positions.to(A.device))

    # Average across attention heads and question tokens
    scores = A.mean(dim=(0, 1))

    return scores

In [ ]:
for layer in [0, 1, 3, 5]:
    scores = attention_relevance(
        attentions[layer],
        question_positions,
        image_positions
    )

    visualize_relevance(
        image,
        f"Layer {layer}: {question}",
        scores,
        inputs["image_grid_thw"]
    )

In [ ]:
api.set_loops(vlm, 3)

def attention_selector(scores, k=0.5):
    assert 0 < k <= 1
    n_keep = max(1, int(len(scores) * k))
    return scores.topk(n_keep).indices.sort().values

for layer in range(6):
    scores = attention_relevance(attentions[layer], question_positions, image_positions)
    pruned = api.prune_inputs(vlm, inputs, vision_outputs, attention_selector(scores, k=0.5))
    print(f"\nLayer: {layer}")
    print(api.generate(vlm, pruned, MAX_NEW_TOKENS).text)